<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/glasslego/ml-deep-learning-study/blob/main/src/deep_learning_basic/04_cnn_basics.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />구글 코랩에서 실행하기</a>
  </td>
</table>

# CNN(Convolutional Neural Network) 기초 - 이미지 처리의 핵심

CNN은 이미지 인식에 혁명을 일으킨 딥러닝 모델입니다. 이 노트북에서는 CNN의 핵심 개념을 쉽게 이해하고, 실제로 이미지를 처리하는 예제를 실습합니다.

In [ ]:
# PyTorch 임포트 (Google Colab T4 환경)
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# 나눔고딕 폰트 설치
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

# 폰트 설정
plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)  # 마이너스 기호 깨짐 방지

print(f"PyTorch 버전: {torch.__version__}")

# GPU 설정 (Google Colab T4 환경)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using device: {device}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA 버전: {torch.version.cuda}")
else:
    device = torch.device("cpu")
    print(f"Using device: {device}")

# 시드 설정 (재현 가능한 결과를 위해)
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

## 1. CNN의 핵심 개념 이해하기

### 왜 CNN이 이미지 처리에 좋을까?

일반적인 신경망(Fully Connected)의 문제점:
- 28x28 이미지 = 784개의 입력
- 모든 픽셀이 모든 뉴런과 연결 → 파라미터가 너무 많음
- 이미지의 공간적 구조를 무시함

CNN의 장점:
1. **지역적 패턴 학습**: 작은 필터로 지역적 특징을 감지
2. **파라미터 공유**: 같은 필터를 이미지 전체에 적용
3. **계층적 학습**: 낮은 층에서 간단한 특징, 높은 층에서 복잡한 특징

## 2. CNN의 주요 구성 요소 시각화

In [ ]:
# 간단한 이미지 생성 (체크보드 패턴)
def create_checkboard(size=8, square_size=1):
    board = np.zeros((size, size))
    for i in range(0, size, square_size*2):
        for j in range(0, size, square_size*2):
            board[i:i+square_size, j:j+square_size] = 1
            board[i+square_size:i+square_size*2, j+square_size:j+square_size*2] = 1
    return board

# 다양한 필터 정의
filters = {
    '수직 엣지': np.array([[-1, 0, 1],
                         [-1, 0, 1],
                         [-1, 0, 1]]),
    
    '수평 엣지': np.array([[-1, -1, -1],
                         [ 0,  0,  0],
                         [ 1,  1,  1]]),
    
    '블러': np.array([[1, 1, 1],
                    [1, 1, 1],
                    [1, 1, 1]]) / 9,
    
    '샤프닝': np.array([[ 0, -1,  0],
                      [-1,  5, -1],
                      [ 0, -1,  0]])
}

# 합성곱 연산 시각화
def visualize_convolution(image, kernel, stride=1):
    """
    합성곱 연산을 단계별로 시각화합니다.
    """
    h, w = image.shape
    kh, kw = kernel.shape
    oh = (h - kh) // stride + 1
    ow = (w - kw) // stride + 1
    output = np.zeros((oh, ow))
    
    # 합성곱 연산
    for i in range(oh):
        for j in range(ow):
            region = image[i*stride:i*stride+kh, j*stride:j*stride+kw]
            output[i, j] = np.sum(region * kernel)
    
    return output

# 체크보드 이미지 생성
checkboard = create_checkboard(16, 2)

# 각 필터 적용 결과 시각화
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

# 원본 이미지
axes[0, 0].imshow(checkboard, cmap='gray')
axes[0, 0].set_title('원본 이미지')
axes[0, 0].axis('off')
axes[1, 0].axis('off')

# 각 필터 적용
for idx, (name, kernel) in enumerate(filters.items()):
    col = idx + 1
    
    # 필터 시각화
    axes[0, col].imshow(kernel, cmap='RdBu', vmin=-1, vmax=1)
    axes[0, col].set_title(f'{name} 필터')
    axes[0, col].axis('off')
    
    # 합성곱 결과
    result = visualize_convolution(checkboard, kernel)
    axes[1, col].imshow(result, cmap='gray')
    axes[1, col].set_title(f'{name} 결과')
    axes[1, col].axis('off')

plt.tight_layout()
plt.suptitle('합성곱 필터의 효과', fontsize=16, y=1.02)
plt.show()

## 3. 풀링(Pooling) 이해하기

In [ ]:
def max_pooling(image, pool_size=2, stride=2):
    """
    Max Pooling 연산을 수행합니다.
    """
    h, w = image.shape
    oh = (h - pool_size) // stride + 1
    ow = (w - pool_size) // stride + 1
    output = np.zeros((oh, ow))
    
    for i in range(oh):
        for j in range(ow):
            region = image[i*stride:i*stride+pool_size, j*stride:j*stride+pool_size]
            output[i, j] = np.max(region)
    
    return output

def average_pooling(image, pool_size=2, stride=2):
    """
    Average Pooling 연산을 수행합니다.
    """
    h, w = image.shape
    oh = (h - pool_size) // stride + 1
    ow = (w - pool_size) // stride + 1
    output = np.zeros((oh, ow))
    
    for i in range(oh):
        for j in range(ow):
            region = image[i*stride:i*stride+pool_size, j*stride:j*stride+pool_size]
            output[i, j] = np.mean(region)
    
    return output

# 샘플 이미지 생성 (숫자 패턴)
sample_image = np.array([[1, 2, 3, 4],
                        [5, 6, 7, 8],
                        [9, 10, 11, 12],
                        [13, 14, 15, 16]])

# 풀링 적용
max_pooled = max_pooling(sample_image)
avg_pooled = average_pooling(sample_image)

# 시각화
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 4))

# 원본
im1 = ax1.imshow(sample_image, cmap='viridis')
ax1.set_title('원본 (4x4)', fontsize=14)
ax1.grid(True, color='white', linewidth=2)
ax1.set_xticks(np.arange(-0.5, 4, 1))
ax1.set_yticks(np.arange(-0.5, 4, 1))
# 값 표시
for i in range(4):
    for j in range(4):
        ax1.text(j, i, str(int(sample_image[i, j])), 
                ha='center', va='center', color='white', fontsize=12)

# Max Pooling
im2 = ax2.imshow(max_pooled, cmap='viridis')
ax2.set_title('Max Pooling (2x2)', fontsize=14)
ax2.grid(True, color='white', linewidth=2)
ax2.set_xticks(np.arange(-0.5, 2, 1))
ax2.set_yticks(np.arange(-0.5, 2, 1))
for i in range(2):
    for j in range(2):
        ax2.text(j, i, str(int(max_pooled[i, j])), 
                ha='center', va='center', color='white', fontsize=12)

# Average Pooling
im3 = ax3.imshow(avg_pooled, cmap='viridis')
ax3.set_title('Average Pooling (2x2)', fontsize=14)
ax3.grid(True, color='white', linewidth=2)
ax3.set_xticks(np.arange(-0.5, 2, 1))
ax3.set_yticks(np.arange(-0.5, 2, 1))
for i in range(2):
    for j in range(2):
        ax3.text(j, i, f'{avg_pooled[i, j]:.1f}', 
                ha='center', va='center', color='white', fontsize=12)

plt.tight_layout()
plt.show()

print("풀링의 효과:")
print("- 이미지 크기 감소: 4x4 → 2x2")
print("- Max Pooling: 각 영역에서 가장 큰 값 선택 (특징 강조)")
print("- Average Pooling: 각 영역의 평균값 계산 (부드러운 다운샘플링)")

## 4. CNN 아키텍처 구현

In [ ]:
class SimpleCNN(nn.Module):
    """
    간단한 CNN 구조
    Conv -> ReLU -> Pool -> Conv -> ReLU -> Pool -> FC
    """
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        
        # 첫 번째 합성곱 블록
        self.conv1 = nn.Conv2d(in_channels=3,      # RGB 3채널
                              out_channels=16,     # 16개 필터
                              kernel_size=3,       # 3x3 필터
                              padding=1)           # same padding
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)  # 2x2 풀링
        
        # 두 번째 합성곱 블록
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        
        # 세 번째 합성곱 블록
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(2, 2)
        
        # 완전연결층
        self.fc1 = nn.Linear(64 * 4 * 4, 128)  # 32x32 -> 4x4 after 3 poolings
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, num_classes)
    
    def forward(self, x):
        # 합성곱 블록들
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = self.pool3(F.relu(self.conv3(x)))
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # 완전연결층
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

# 모델 생성 및 구조 확인
model = SimpleCNN(num_classes=10).to(device)
print("SimpleCNN 구조:")
print(model)
print(f"\n총 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

## 5. CIFAR-10 데이터셋으로 실습

In [ ]:
# 데이터 변환 정의
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),  # 50% 확률로 좌우 반전
    transforms.RandomCrop(32, padding=4),    # 랜덤 크롭
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                        std=[0.229, 0.224, 0.225])  # ImageNet 통계
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                        std=[0.229, 0.224, 0.225])
])

# CIFAR-10 데이터셋 로드
train_dataset = datasets.CIFAR10(root='./data', train=True, 
                                download=True, transform=transform_train)
test_dataset = datasets.CIFAR10(root='./data', train=False, 
                               download=True, transform=transform_test)

# DataLoader 생성
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, 
                         shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, 
                        shuffle=False, num_workers=2)

# CIFAR-10 클래스 이름
classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']

print(f"학습 데이터: {len(train_dataset):,}개")
print(f"테스트 데이터: {len(test_dataset):,}개")
print(f"이미지 크기: 32x32x3 (RGB)")
print(f"클래스: {', '.join(classes)}")

## 6. 데이터 샘플 시각화

In [ ]:
# 정규화 역변환 함수
def denormalize(tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    for t, m, s in zip(tensor, mean, std):
        t.mul_(s).add_(m)
    return tensor

# 샘플 이미지 시각화
fig, axes = plt.subplots(2, 5, figsize=(12, 6))
axes = axes.ravel()

# 원본 변환 사용 (augmentation 없음)
sample_dataset = datasets.CIFAR10(root='./data', train=True, 
                                 download=False, transform=transform_test)

for i in range(10):
    img, label = sample_dataset[i]
    img = denormalize(img.clone())
    img = img.permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    
    axes[i].imshow(img)
    axes[i].set_title(f'{classes[label]}', fontsize=12)
    axes[i].axis('off')

plt.tight_layout()
plt.suptitle('CIFAR-10 샘플 이미지', fontsize=16, y=1.02)
plt.show()

## 7. CNN 모델 학습

In [ ]:
def train_epoch(model, train_loader, optimizer, criterion, epoch):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        if i % 100 == 99:
            print(f'[Epoch {epoch}, Batch {i+1}] Loss: {running_loss/100:.3f}, '
                  f'Acc: {100.*correct/total:.2f}%')
            running_loss = 0.0
    
    return 100. * correct / total

def test(model, test_loader, criterion):
    model.eval()
    test_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    avg_loss = test_loss / len(test_loader)
    accuracy = 100. * correct / total
    
    return avg_loss, accuracy

# 학습 설정
model = SimpleCNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

# 학습
print("CNN 학습 시작...")
train_accs = []
test_accs = []

num_epochs = 10
for epoch in range(1, num_epochs + 1):
    print(f"\n=== Epoch {epoch}/{num_epochs} ===")
    
    train_acc = train_epoch(model, train_loader, optimizer, criterion, epoch)
    test_loss, test_acc = test(model, test_loader, criterion)
    
    train_accs.append(train_acc)
    test_accs.append(test_acc)
    
    print(f'\nTrain Accuracy: {train_acc:.2f}%')
    print(f'Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.2f}%')
    
    scheduler.step()

## 8. 학습 결과 시각화

In [ ]:
# 학습 곡선 그리기
plt.figure(figsize=(10, 6))
plt.plot(range(1, num_epochs + 1), train_accs, 'b-', label='Train Accuracy')
plt.plot(range(1, num_epochs + 1), test_accs, 'r-', label='Test Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('CNN 학습 곡선')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 9. 예측 결과 시각화

In [ ]:
def visualize_predictions(model, test_loader, num_images=20):
    model.eval()
    images, labels = next(iter(test_loader))
    images, labels = images[:num_images].to(device), labels[:num_images]
    
    with torch.no_grad():
        outputs = model(images)
        _, predictions = outputs.max(1)
        predictions = predictions.cpu()
    
    fig, axes = plt.subplots(4, 5, figsize=(12, 10))
    axes = axes.ravel()
    
    for i in range(num_images):
        img = images[i].cpu()
        img = denormalize(img.clone())
        img = img.permute(1, 2, 0).numpy()
        img = np.clip(img, 0, 1)
        
        axes[i].imshow(img)
        axes[i].axis('off')
        
        color = 'green' if predictions[i] == labels[i] else 'red'
        axes[i].set_title(f'실제: {classes[labels[i]]}\n'
                         f'예측: {classes[predictions[i]]}',
                         color=color, fontsize=10)
    
    plt.tight_layout()
    plt.suptitle('CNN 예측 결과', fontsize=16, y=1.02)
    plt.show()

visualize_predictions(model, test_loader)

## 10. 필터 시각화

In [ ]:
def visualize_filters(model):
    """
    첫 번째 합성곱층의 필터를 시각화합니다.
    """
    # 첫 번째 합성곱층의 가중치 가져오기
    first_conv_weights = model.conv1.weight.data.cpu()
    
    # 16개의 필터 시각화 (각 필터는 3x3x3)
    fig, axes = plt.subplots(4, 4, figsize=(8, 8))
    axes = axes.ravel()
    
    for i in range(16):
        # RGB 채널의 필터를 평균내어 시각화
        filter_rgb = first_conv_weights[i].permute(1, 2, 0).numpy()
        # 정규화
        filter_rgb = (filter_rgb - filter_rgb.min()) / (filter_rgb.max() - filter_rgb.min())
        
        axes[i].imshow(filter_rgb)
        axes[i].set_title(f'Filter {i+1}', fontsize=8)
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.suptitle('첫 번째 합성곱층의 학습된 필터', fontsize=14, y=1.02)
    plt.show()

visualize_filters(model)

## 11. 특징맵(Feature Map) 시각화

In [ ]:
def visualize_feature_maps(model, image):
    """
    각 층의 특징맵을 시각화합니다.
    """
    model.eval()
    
    # Hook을 사용하여 중간 출력 캡처
    activations = []
    
    def hook_fn(module, input, output):
        activations.append(output)
    
    # 각 합성곱층에 hook 등록
    hooks = []
    hooks.append(model.conv1.register_forward_hook(hook_fn))
    hooks.append(model.conv2.register_forward_hook(hook_fn))
    hooks.append(model.conv3.register_forward_hook(hook_fn))
    
    # 이미지 통과
    with torch.no_grad():
        _ = model(image.unsqueeze(0).to(device))
    
    # Hook 제거
    for hook in hooks:
        hook.remove()
    
    # 각 층의 특징맵 시각화
    layer_names = ['Conv1 (16 channels)', 'Conv2 (32 channels)', 'Conv3 (64 channels)']
    
    for layer_idx, (activation, layer_name) in enumerate(zip(activations, layer_names)):
        activation = activation.cpu().squeeze(0)
        num_channels = activation.shape[0]
        
        # 처음 16개 채널만 시각화
        num_to_show = min(16, num_channels)
        fig, axes = plt.subplots(4, 4, figsize=(10, 10))
        axes = axes.ravel()
        
        for i in range(num_to_show):
            feature_map = activation[i].numpy()
            axes[i].imshow(feature_map, cmap='viridis')
            axes[i].set_title(f'Channel {i+1}', fontsize=8)
            axes[i].axis('off')
        
        # 사용하지 않는 subplot 숨기기
        for i in range(num_to_show, 16):
            axes[i].axis('off')
        
        plt.tight_layout()
        plt.suptitle(f'{layer_name} 특징맵', fontsize=14, y=1.02)
        plt.show()

# 테스트 이미지로 특징맵 시각화
test_image, test_label = test_dataset[0]
print(f"테스트 이미지 클래스: {classes[test_label]}")

# 원본 이미지 표시
plt.figure(figsize=(4, 4))
img = denormalize(test_image.clone())
img = img.permute(1, 2, 0).numpy()
img = np.clip(img, 0, 1)
plt.imshow(img)
plt.title(f'입력 이미지: {classes[test_label]}')
plt.axis('off')
plt.show()

# 특징맵 시각화
visualize_feature_maps(model, test_image)

## 12. 데이터 증강(Data Augmentation)의 효과

In [ ]:
# 다양한 데이터 증강 기법 시각화
augmentations = [
    ('원본', transforms.Compose([transforms.ToTensor()])),
    ('수평 반전', transforms.Compose([
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor()
    ])),
    ('랜덤 크롭', transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor()
    ])),
    ('회전', transforms.Compose([
        transforms.RandomRotation(15),
        transforms.ToTensor()
    ])),
    ('색상 변화', transforms.Compose([
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.ToTensor()
    ])),
]

# 원본 이미지 가져오기
original_dataset = datasets.CIFAR10(root='./data', train=True, download=False)
original_image, label = original_dataset[0]  # PIL Image

fig, axes = plt.subplots(1, 5, figsize=(15, 3))

for idx, (name, transform) in enumerate(augmentations):
    # 변환 적용
    transformed = transform(original_image)
    img = transformed.permute(1, 2, 0).numpy()
    
    axes[idx].imshow(img)
    axes[idx].set_title(name, fontsize=12)
    axes[idx].axis('off')

plt.tight_layout()
plt.suptitle(f'데이터 증강 기법들 ({classes[label]})', fontsize=16, y=1.05)
plt.show()

## 13. 실제 이미지로 테스트

In [ ]:
def create_test_images():
    """
    테스트용 간단한 이미지들을 생성합니다.
    """
    # 32x32 RGB 이미지 생성
    test_images = []
    
    # 1. 비행기 모양 (삼각형)
    img1 = np.ones((32, 32, 3), dtype=np.uint8) * 135  # 회색 배경
    # 삼각형 그리기
    for i in range(16):
        for j in range(16-i, 17+i):
            img1[i+8, j] = [200, 200, 255]  # 연한 파란색
    test_images.append(('비행기 모양', img1))
    
    # 2. 자동차 모양 (사각형)
    img2 = np.ones((32, 32, 3), dtype=np.uint8) * 135
    img2[12:20, 8:24] = [255, 0, 0]  # 빨간색 차체
    img2[20:24, 10:22] = [0, 0, 0]   # 검은색 바퀴
    test_images.append(('자동차 모양', img2))
    
    # 3. 새 모양 (V자)
    img3 = np.ones((32, 32, 3), dtype=np.uint8) * 200  # 밝은 배경
    # V자 그리기
    for i in range(10):
        img3[10+i, 16-i] = [100, 50, 0]  # 갈색
        img3[10+i, 16+i] = [100, 50, 0]
    test_images.append(('새 모양', img3))
    
    return test_images

def test_custom_images(model, images):
    """
    사용자 정의 이미지를 테스트합니다.
    """
    model.eval()
    
    fig, axes = plt.subplots(1, len(images), figsize=(12, 4))
    
    for idx, (name, img) in enumerate(images):
        # PIL 이미지로 변환
        pil_img = Image.fromarray(img)
        
        # 전처리
        tensor_img = transform_test(pil_img).unsqueeze(0).to(device)
        
        # 예측
        with torch.no_grad():
            output = model(tensor_img)
            probabilities = F.softmax(output, dim=1).cpu().numpy()[0]
            pred_class = output.argmax(1).item()
        
        # 시각화
        axes[idx].imshow(img)
        axes[idx].set_title(f'{name}\n예측: {classes[pred_class]} '
                           f'({probabilities[pred_class]*100:.1f}%)',
                           fontsize=10)
        axes[idx].axis('off')
        
        # 상위 3개 예측 출력
        top3_idx = np.argsort(probabilities)[-3:][::-1]
        print(f"\n{name} 예측 결과:")
        for i, idx_class in enumerate(top3_idx):
            print(f"  {i+1}. {classes[idx_class]}: {probabilities[idx_class]*100:.1f}%")
    
    plt.tight_layout()
    plt.show()

# 테스트 이미지 생성 및 예측
custom_images = create_test_images()
test_custom_images(model, custom_images)

## 14. CNN vs 일반 신경망 비교

In [ ]:
class SimpleFC(nn.Module):
    """
    CNN과 비교를 위한 일반 신경망
    """
    def __init__(self, num_classes=10):
        super(SimpleFC, self).__init__()
        self.fc1 = nn.Linear(32 * 32 * 3, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, num_classes)
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x

# 모델 비교
fc_model = SimpleFC().to(device)
cnn_model = SimpleCNN().to(device)

print("=== 모델 파라미터 수 비교 ===")
print(f"일반 신경망 (FC): {sum(p.numel() for p in fc_model.parameters()):,} 파라미터")
print(f"합성곱 신경망 (CNN): {sum(p.numel() for p in cnn_model.parameters()):,} 파라미터")
print(f"\nCNN이 FC보다 약 {sum(p.numel() for p in fc_model.parameters()) / sum(p.numel() for p in cnn_model.parameters()):.1f}배 적은 파라미터 사용")

## 요약

### CNN의 핵심 개념:

1. **합성곱(Convolution)**:
   - 작은 필터로 이미지를 스캔하며 특징 추출
   - 파라미터 공유로 효율적인 학습
   - 위치 불변성 학습

2. **풀링(Pooling)**:
   - 이미지 크기 감소
   - 중요한 특징 보존
   - 계산량 감소

3. **계층적 특징 학습**:
   - 낮은 층: 엣지, 코너 등 단순 특징
   - 중간 층: 텍스처, 패턴
   - 높은 층: 객체 부분, 전체 객체

### CNN의 장점:
- 이미지의 공간적 구조 활용
- 적은 파라미터로 효율적 학습
- 위치/크기 변화에 강인함
- 다양한 컴퓨터 비전 작업에 활용 가능

### 실습 내용:
- CIFAR-10 데이터셋으로 실제 이미지 분류
- 필터와 특징맵 시각화
- 데이터 증강 기법
- 약 75-80% 정확도 달성 (더 깊은 모델과 긴 학습으로 90%+ 가능)